# Validation Tests

This notebook tests the 7 input-level validation checks in `_validate_config()`.

| # | Scenario | Expected Result |
|---|----------|-----------------|
| 1 | Missing required columns | `ConfigurationError` |
| 2 | Empty required values | `ValidationError` |
| 3 | Invalid characters in target names | `ValidationError` |
| 4 | Both include_columns and exclude_columns | `ValidationError` |
| 5 | Invalid cron expression | `ValidationError` |
| 6 | Duplicate source rows | Warning (proceeds successfully) |
| 7 | SCD Type 2 + CDC recommendation | Info log (proceeds successfully) |

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%pip install pyyaml

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

username = w.current_user.me().user_name
workspace_host = w.config.host

WORKSPACE_HOST = workspace_host
ROOT_PATH = f'/Users/{username}/.bundle/${{bundle.name}}/${{bundle.target}}'

In [ ]:
import sys
import os
import logging

sys.path.append(os.path.abspath('../../src'))

from tapworks.core.runner import run_pipeline_generation

# Enable logging to see warnings and info messages
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('tapworks')
logger.setLevel(logging.INFO)

targets = {
    'dev': {
        'workspace_host': WORKSPACE_HOST,
        'root_path': ROOT_PATH
    }
}

## Test 1: Missing Required Columns

CSV is missing `target_table_name` and `connection_name`.

**Expected:** `ConfigurationError` listing the missing columns.

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='01_missing_required_columns/pipeline_config.csv',
        output_dir='01_missing_required_columns/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 2: Empty Required Values

Row 2 has empty `source_table_name`, row 3 has empty `target_schema`.

**Expected:** `ValidationError` identifying which columns are empty in which rows.

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='02_empty_required_values/pipeline_config.csv',
        output_dir='02_empty_required_values/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 3: Invalid Characters in Target Names

Row 2 has `target_table_name='my-customers!'` (hyphens and exclamation mark).
Row 3 has `target_table_name='order items'` (space).

**Expected:** `ValidationError` listing the invalid values.

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='03_invalid_target_characters/pipeline_config.csv',
        output_dir='03_invalid_target_characters/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 4: Both include_columns and exclude_columns Set

Uses Salesforce connector. Row 2 (Contact) has both `include_columns` and `exclude_columns` set.

**Expected:** `ValidationError` — these are mutually exclusive.

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='salesforce',
        input_source='04_both_include_exclude/pipeline_config.csv',
        output_dir='04_both_include_exclude/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 5: Invalid Cron Expression

Row 2 has `schedule='*/15 * *'` (only 3 fields instead of 5).
Row 3 has `schedule='not a cron'`.

**Expected:** `ValidationError` — invalid cron format.

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='05_invalid_cron/pipeline_config.csv',
        output_dir='05_invalid_cron/deployment',
        targets=targets,
    )
    print('UNEXPECTED: No error raised')
except Exception as e:
    print(f'EXPECTED ERROR ({type(e).__name__}): {e}')

## Test 6: Duplicate Source Rows (Warning Only)

Rows 1 and 2 are identical (same Products table).

**Expected:** Warning logged about duplicates, but pipeline generation **succeeds**. Check the logs for the warning message.

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='06_duplicate_source_rows/pipeline_config.csv',
        output_dir='06_duplicate_source_rows/deployment',
        targets=targets,
    )
    print(f'SUCCESS: Processed {len(result)} rows (check logs above for duplicate warning)')
    print(f'Pipeline groups: {result["pipeline_group"].nunique()}')
except Exception as e:
    print(f'UNEXPECTED ERROR ({type(e).__name__}): {e}')

## Test 7: SCD Type 2 + CDC Recommendation (Info Only)

Row 1 uses `SCD_TYPE_1`, rows 2-3 use `SCD_TYPE_2`.

**Expected:** Info log recommending CDC enablement for the 2 SCD_TYPE_2 rows. Pipeline generation **succeeds**. Check the logs for the info message.

In [ ]:
try:
    result = run_pipeline_generation(
        connector_name='sql_server',
        input_source='07_scd_type2_cdc_recommendation/pipeline_config.csv',
        output_dir='07_scd_type2_cdc_recommendation/deployment',
        targets=targets,
    )
    print(f'SUCCESS: Processed {len(result)} rows (check logs above for CDC recommendation)')
    print(f'Pipeline groups: {result["pipeline_group"].nunique()}')
    print(f'SCD types used: {result["scd_type"].value_counts().to_dict()}')
except Exception as e:
    print(f'UNEXPECTED ERROR ({type(e).__name__}): {e}')